In [0]:
select * from dev.bronze.emp

emp_id,emp_name,dept_code,salary
1001,Shubham,D101,10000.0
1002,Rahul,D101,12000.0
1003,Amit,D102,5000.0
1004,Priya,D102,12000.0
1005,Smith,D101,10000.0


In [0]:
-- Create updates table
create table if not exists dev.bronze.emp_updates (
    emp_id int,
    emp_name string,
    dept_code string,
    salary double
)

In [0]:
INSERT INTO dev.bronze.emp_updates (emp_id, emp_name, dept_code, salary)
VALUES
    (1001, 'Shubham', 'D101', 75000),
    (1002, 'Rahul', 'D101', 60000),
    (1003, 'Amit', 'D102', 55000)

num_affected_rows,num_inserted_rows
3,3


In [0]:
-- Add new record into emp_updates table
INSERT INTO dev.bronze.emp_updates (emp_id, emp_name, dept_code, salary)
VALUES
    (1006, 'Rohini', 'D103', 70000)

num_affected_rows,num_inserted_rows
1,1


In [0]:
select * from dev.bronze.emp_updates

emp_id,emp_name,dept_code,salary
1001,Shubham,D101,75000.0
1002,Rahul,D101,60000.0
1003,Amit,D102,55000.0
1006,Rohini,D103,70000.0


In [0]:
-- Merge data to update existing records and imsert new records
MERGE INTO dev.bronze.emp t
using dev.bronze.emp_updates s      
on s.emp_id = t.emp_id
when matched THEN
update SET
t.salary = s.salary
when not MATCHED THEN
INSERT (emp_id, emp_name, dept_code, salary) VALUES (s.emp_id, s.emp_name, s.dept_code, s.salary);


num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
4,3,0,1


In [0]:
select * from dev.bronze.emp

emp_id,emp_name,dept_code,salary
1001,Shubham,D101,75000.0
1002,Rahul,D101,60000.0
1003,Amit,D102,55000.0
1006,Rohini,D103,70000.0
1004,Priya,D102,12000.0
1005,Smith,D101,10000.0


In [0]:
-- Merge data to update existing records and imsert new records and not matched source delete
MERGE INTO dev.bronze.emp t
using dev.bronze.emp_updates s      
on s.emp_id = t.emp_id
when matched THEN
update SET
t.salary = s.salary
when not MATCHED THEN
INSERT (emp_id, emp_name, dept_code, salary) VALUES (s.emp_id, s.emp_name, s.dept_code, s.salary)
WHEN NOT MATCHED BY SOURCE
THEN DELETE;

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
6,4,2,0


In [0]:
select * from dev.bronze.emp

emp_id,emp_name,dept_code,salary,is_active
1001,Shubham,D101,75000.0,null
1002,Rahul,D101,60000.0,null
1003,Amit,D102,55000.0,null
1006,Rohini,D103,70000.0,null


In [0]:
Alter table dev.bronze.emp add column is_active string;

In [0]:
-- Add new record into emp_updates table
INSERT INTO dev.bronze.emp (emp_id, emp_name, dept_code, salary)
VALUES
    (1009, 'AKSHAY', 'D104', 7000)

num_affected_rows,num_inserted_rows
1,1


In [0]:
-- Merge data to update existing records and imsert new records and soft delete
MERGE INTO dev.bronze.emp t
using dev.bronze.emp_updates s      
on s.emp_id = t.emp_id
when matched THEN
update SET
t.salary = s.salary,
t.is_active = 'Y'
when not MATCHED THEN
INSERT (emp_id, emp_name, dept_code, salary,is_active) VALUES (s.emp_id, s.emp_name, s.dept_code, s.salary,'Y')
when not MATCHED by source then
UPDATE SET t.is_active = 'N';

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
6,6,0,0


In [0]:
select * from dev.bronze.emp

emp_id,emp_name,dept_code,salary,is_active
1001,Shubham,D101,75000.0,Y
1002,Rahul,D101,60000.0,Y
1003,Amit,D102,55000.0,Y
1006,Rohini,D103,70000.0,Y
1008,Sunny,D103,7000.0,Y
1009,AKSHAY,D104,7000.0,N


# 📝 Databricks SQL — MERGE Operations Practice Notes

---

## 📌 What I Practiced

1. **Created an updates table** (`dev.bronze.emp_updates`) to stage incoming employee data.
2. **Inserted new records** into the updates table for change capture.
3. **Basic MERGE (Update + Insert)** — updated `salary` for matching records and inserted new records from the source.
4. **MERGE with Hard Delete** — added `WHEN NOT MATCHED BY SOURCE THEN DELETE` to remove records that no longer exist in the source.
5. **Added an `is_active` column** to the target table for soft-delete tracking.
6. **MERGE with Soft Delete** — set `is_active = 'Y'` for matched/new records and `is_active = 'N'` for records not matched by source (instead of physically deleting).

---

## ⭐ Important Points

| Concept | Key Takeaway |
|---|---|
| **MERGE INTO** | Combines UPDATE, INSERT, and DELETE in a single atomic operation on a target table. |
| **WHEN MATCHED** | Fires when source and target rows match on the join key — typically used for UPDATE. |
| **WHEN NOT MATCHED (BY TARGET)** | Fires when a source row has no match in the target — typically used for INSERT. |
| **WHEN NOT MATCHED BY SOURCE** | Fires when a target row has no match in the source — used for DELETE or soft-delete UPDATE. |
| **Hard Delete vs Soft Delete** | Hard delete physically removes rows; soft delete marks rows with a flag (e.g., `is_active = 'N'`). |
| **Atomicity** | The entire MERGE is one transaction — either all changes apply or none do. |
| **Source vs Target** | `t` = target (table being merged INTO), `s` = source (data driving changes). |
| **Schema Evolution** | Adding columns (e.g., `is_active`) requires `ALTER TABLE` before using them in MERGE. |

---

## ❓ Important Questions

1. **What is the difference between `WHEN NOT MATCHED` and `WHEN NOT MATCHED BY SOURCE`?**
   - `WHEN NOT MATCHED` (short for BY TARGET): source rows with no target match → used to INSERT new records.
   - `WHEN NOT MATCHED BY SOURCE`: target rows with no source match → used to DELETE or soft-delete stale records.

2. **When should you use a hard delete vs a soft delete?**
   - Use **hard delete** when the data is truly obsolete and audit/history is not needed.
   - Use **soft delete** when you need to retain history, comply with audit requirements, or allow reactivation.

3. **Can you have multiple `WHEN MATCHED` clauses?**
   - Yes, but only one can have an UPDATE (or DELETE) action without a condition; additional clauses must have conditional predicates.

4. **What happens if the source has duplicate keys?**
   - MERGE will fail with an error — the source must have at most one row per match key.

5. **Is MERGE atomic?**
   - Yes. All matched/not-matched actions execute as a single transaction. If any part fails, the entire operation rolls back.

6. **Why add an `is_active` column instead of deleting?**
   - It preserves the row for reporting/auditing, allows filtering active vs inactive records, and avoids irreversible data loss.

---

## 📊 Summary

| Step | Operation | Effect on Target |
|---|---|---|
| 1 | Create `emp_updates` table | Staging table for incoming changes |
| 2 | Insert records into `emp_updates` | Load source data |
| 3 | Basic MERGE (update + insert) | Update salary for matches; insert new rows |
| 4 | MERGE + hard delete | Delete target rows not present in source |
| 5 | `ALTER TABLE` add `is_active` | Track active/inactive state |
| 6 | MERGE + soft delete | Set `is_active = 'Y'` for active, `'N'` for stale |

### Key Learning
> MERGE is a powerful single-statement upsert (update + insert) that can also handle deletes. Choosing between hard and soft deletes depends on whether you need to preserve historical/audit data. Always ensure the source has unique keys to avoid merge failures.